<a href="https://colab.research.google.com/github/satyaudaybandaru/Gemma-3-4B-FunctionCall/blob/main/Phase_2_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [ ]:
%%capture
!pip install --upgrade unsloth transformers huggingface_hub trl

In [ ]:
from unsloth import FastModel
import torch


model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

model.config.vocab_size = model.config.text_config.vocab_size # For newer versions support

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Load Phase-1 Finetuned model from checkpoint

In [ ]:
from transformers import AutoTokenizer
tk = AutoTokenizer.from_pretrained('/content/drive/MyDrive/gemma3-4b-toolcall/checkpoint-80')

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/gemma3-4b-toolcall/checkpoint-80',is_trainable=True)

#### Verify trainable params to ensure everything loaded correctly

In [ ]:
model.print_trainable_parameters()

#### Load prepared dataset again

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive.flush_and_unmount()

In [ ]:
from datasets import load_from_disk
filtered_dataset = load_from_disk("/content/drive/MyDrive/gemma3-dataset")

In [ ]:
filtered_dataset['text'][0]

#### We train on both regular conversations and tool-calling data to prevent catastrophic forgetting.

In [ ]:
MODEL_START = tk(
    "<start_of_turn>model\n",
    add_special_tokens=False
)["input_ids"]

END_TURN = tk(
    "<end_of_turn>",
    add_special_tokens=False
)["input_ids"]

print(MODEL_START)
print(END_TURN)

In [ ]:
# mask all except model turn (normal convo + tool calls)

def create_labels(input_ids):
    labels = [-100] * len(input_ids)

    i = 0
    while i < len(input_ids):

        if input_ids[i:i+3] == MODEL_START:

            start = i + 3

            j = start

            while j < len(input_ids):

                if input_ids[j:j+len(END_TURN)] == END_TURN:

                    labels[start:j+len(END_TURN)] = input_ids[start:j+len(END_TURN)]
                    i = j
                    break

                j += 1

        i += 1

    return labels

In [ ]:
# Mask all except <tool_call>

TOOL_MODEL_START = tk(
    "<tool_call>",
    add_special_tokens=False
)["input_ids"]


print(TOOL_MODEL_START)


def create_tool_labels(input_ids):
    labels = [-100] * len(input_ids)

    i = 0

    while i < len(input_ids):
        if input_ids[i:i+5] == TOOL_MODEL_START:

            start = i

            j = start+5
            while j < len(input_ids):
                if input_ids[j:j+len(END_TURN)] == END_TURN:

                    labels[start:j+len(END_TURN)] = input_ids[start:j+len(END_TURN)]
                    i = j
                    break

                j += 1

        i += 1

    return labels

In [ ]:
def preprocess(example):
    encoded = tk(
        example["text"],
        truncation=True,
        max_length=4096,
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": create_labels(encoded["input_ids"]),
    }

In [ ]:
def preprocess_tool(example):
    encoded = tk(
        example["text"],
        truncation=True,
        max_length=4096,
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": create_tool_labels(encoded["input_ids"]),
    }

In [ ]:
processed_dataset = filtered_dataset.select(range(250)).map(
    preprocess,
    remove_columns=filtered_dataset.column_names,
    num_proc=4,
)

processed_tool_dataset = filtered_dataset.select(range(251,500)).map(
    preprocess_tool,
    remove_columns=filtered_dataset.column_names,
    num_proc=4,
)

In [ ]:
from datasets import concatenate_datasets
combined_dataset = (concatenate_datasets([processed_dataset, processed_tool_dataset])).shuffle(seed=3407)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tk,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

In [ ]:
# Stage 2 Trainer

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tk,
    train_dataset = combined_dataset,
    eval_dataset = None, # Can set up evaluation!
    data_collator=data_collator,
    args = SFTConfig(
        #dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 50,
        learning_rate = 5e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        output_dir = '/content/drive/MyDrive/gemma3-4b-stage2toolcall/',
        save_steps = 10,
        save_total_limit = 7
    ),
)

Let's print the 100th row again.  Notice how the sample only has a single `<bos>` as expected!

In [ ]:
#Check for different datapoints

datapoint = 398

tk.decode([tk.pad_token_id if x == -100 else x for x in combined_dataset[datapoint]["labels"]]).replace(tk.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
from transformers.models.gemma3.configuration_gemma3 import Gemma3Config

Gemma3Config.vocab_size = property(
    lambda self: self.text_config.vocab_size
)

In [ ]:
tk.pad = tk.tokenizer.pad

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
tools = [
  {
        "name": "get_current_time",
        "description": "Returns current date and time for a timezone.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {"type": "string"}
            }
        }
    },
    {
        "name": "web_search",
        "description": "Searches the web for current information.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string"}
            }
        }
    },
    {
        "name": "calculator",
        "description": "Evaluates mathematical expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            }
        }
    },
    {
        "name": "weather_lookup",
        "description": "Returns current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"}
            }
        }
    },
    {
        "name": "currency_converter",
        "description": "Converts one currency into another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"}
            }
        }
    }
]

messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant. Follow user instructions."}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "the price of 1 gram of 24k gold in Indian rupees"}
        ]
    }
]

inputs = tk.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
)


outputs = model.generate(
    **inputs.to("cuda"),
    max_new_tokens=64,
    temperature=1.0,
    top_p=0.95,
    top_k=64
)

op = tk.batch_decode(outputs, skip_special_tokens=False)

In [ ]:
op[0]

**Continue to Evaluation notebook**